In [1]:
from rich.console import Console
import json
from common.utils import setup

In [2]:
setup()

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
checklist = []
completed = []

In [5]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [6]:
get_checklist_report()

''

In [7]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [9]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [11]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [17]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [18]:
checklist, completed = [], []
loop(messages)

Checklist #1: Set up the distance equation for the two trains.
Checklist #2: Account for the 1-hour head start of the Boston train.
Checklist #3: Solve for the meeting time.
Checklist #4: State the final answer clearly in Rich console markup.

Let `d` be the distance between Boston and New York. Let the trains meet after `t` hours from 2:00 pm. Then:

Boston train distance = `60t`

New York train distance = `80(t-1)` because it starts 1 hour later.

Checklist #1: Set up the distance equation for the two trains.
Checklist #2: Account for the 1-hour head start of the Boston train.
Checklist #3: Solve for the meeting time.
Checklist #4: State the final answer clearly in Rich console markup.

Use the head start directly in the equation:

`60t + 80(t - 1) = d`

Since the actual distance `d` is not given, solve for the meeting time by assuming the trains meet when their 
combined travel equals the distance between cities. The key is that the Boston train travels 1 extra hour before 
the New York train starts.

Checklist #1: Set up the distance equation for the two trains.
Checklist #2: Account for the 1-hour head start of the Boston train.
Checklist #3: Solve for the meeting time.
Checklist #4: State the final answer clearly in Rich console markup.

Because the distance between the cities is not provided, the meeting time cannot be determined uniquely from the 
given information alone.

If the distance were `d`, then the meeting time `t` hours after 2:00 pm would satisfy:

`60t + 80(t - 1) = d`

So `t = (d + 80) / 140`.

Without `d`, there is no single numeric meeting time.

Checklist #1: Set up the distance equation for the two trains.
Checklist #2: Account for the 1-hour head start of the Boston train.
Checklist #3: Solve for the meeting time.
Checklist #4: State the final answer clearly in Rich console markup.

Final answer: The problem does not provide the distance between Boston and New York, so the meeting time cannot be 
determined uniquely.

Formula if the distance is `d` miles:
`60t + 80(t - 1) = d`

` t = (d + 80) / 140 `

They meet `t` hours after 2:00 pm.

Checklist #1: Set up the distance equation for the two trains.
Checklist #2: Account for the 1-hour head start of the Boston train.
Checklist #3: Solve for the meeting time.
Checklist #4: State the final answer clearly in Rich console markup.

Final answer: The meeting time cannot be determined uniquely from the information given, because the distance 
between Boston and New York is not provided.

If the distance is `d` miles, then:

`60t + 80(t - 1) = d`

So,

` t = (d + 80) / 140 `

They meet `t` hours after 2:00 pm.